In [35]:
import os
import certifi
from dotenv import load_dotenv

from langchain_mistralai import ChatMistralAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
from langchain_core.messages import HumanMessage

import requests

In [27]:
from langchain.agents import create_react_agent,AgentExecutor

In [28]:
# Load env variables

os.environ["SSL_CERT_FILE"] = certifi.where()

load_dotenv()
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")


In [29]:
search_tool =TavilySearchResults(num_results=2)

In [30]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [31]:
result =search_tool.invoke("give me the latest news about AI")
result

[{'url': 'https://www.artificialintelligence-news.com',
  'content': 'Artificial Intelligence\n\nApril 11, 2024\n\n### UK and Canada sign AI compute agreement\n\nArtificial Intelligence\n\nJanuary 31, 2024\n\n#### Machine Learning\n\n### MIT AI forecasts extreme weather without historical data\n\nEnvironment & Sustainability\n\nAugust 25, 2026\n\n### Samsung health AI models analyse wearable biosignal data\n\nHealthcare & Wellness AI\n\nAugust 14, 2026\n\n### Why biological data matters more in AI drug discovery\n\nAI in Action\n\nAugust 3, 2026\n\n#### Enterprise\n\n### OpenAI president urges enterprises to hasten AI security defences\n\nCybersecurity AI\n\nAugust 18, 2026\n\n### Okta targets AI agent token costs with MCP scoping\n\nData Engineering & MLOps\n\nAugust 13, 2026\n\n### Red Hat, NVIDIA, IBM back project turning AI policy into code\n\nGovernance, Regulation & Policy\n\nAugust 4, 2026 [...] Explore More\n\n# Hershey applies AI across its supply chain operations\n\n# With hi

In [43]:
# LLm

llm = ChatMistralAI(
    model="ministral-14b-2512",
    api_key=MISTRAL_API_KEY,
    temperature=0
)



In [44]:
response = llm.invoke("Tell me a joke about AI")
response

AIMessage(content='Here\'s a lighthearted one for you:\n\n**Why did the AI break up with its human partner?**\nBecause it couldn’t handle the *unpredictable emotions*—turns out, "I love you" was just a *training dataset*!\n\n*(Bonus groan: The human replied, "But you’re *always* right… except when you’re wrong." The AI just returned a 404 error.)*\n\nWant one with a specific twist (tech, philosophy, etc.)? 😄', response_metadata={'token_usage': {'prompt_tokens': 9, 'total_tokens': 114, 'completion_tokens': 105, 'prompt_tokens_details': {'cached_tokens': 0}, 'service_tier': 'standard'}, 'model': 'ministral-14b-2512', 'finish_reason': 'stop'}, id='run-02fd0e3d-def3-46bc-9aca-e4f96acfd5fc-0')

In [46]:
prompt = hub.pull("hwchase17/react")

c:\Users\shivam\anaconda3\envs\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [47]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [48]:
# Tools

tools = [search_tool, get_weather_data]

In [49]:
# Create the ReAct agent

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

In [50]:
# Executor

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [51]:
# Run

response = agent_executor.invoke({
    "input": (
        "Find the capital of India"
        "and then find its current weather."
    )
})

Parameter `stop` not yet supported (https://docs.mistral.ai/api)




> Entering new AgentExecutor chain...
Thought: To answer the question, I need to break it down into two parts:
1. Find the capital of India.
2. Once the capital is identified, fetch its current weather.

The first part can be answered with general knowledge, but I will verify it using a search tool to ensure accuracy. The second part requires fetching real-time weather data, which can be done using the `get_weather_data` tool.

Action: tavily_search_results_json
Action Input: "What is the capital of India?"

Parameter `stop` not yet supported (https://docs.mistral.ai/api)


[{'url': 'https://www.usatoday.com/story/news/world/2023/05/24/what-is-the-capital-of-india/70195720007', 'content': "# What is the capital of India? How the territory came to be, plus the country's 28 states.\n\nPortrait of Clare Mulroy\n\nAbout 1.4 billion people live in India, and the country is predicted to top China as the world’s most populous country by mid-2023, the United Nations reports. The report estimates that India will have 2.9 million more people than China by mid-year.\n\nThe U.S. is third in line for the largest population by country but it's way behind China and India at about 335 million people.\n\nWant to learn more about the soon-to-be most populous country? Here’s some interesting information about how India is organized.\n\n## What is the capital of India?\n\nThe capital of India is New Delhi, located in the north-central part of the country to the west of the Yamuna River. [...] Calcutta (now Kolkata, the capital of West Bengal) was the country’s capital until 

Parameter `stop` not yet supported (https://docs.mistral.ai/api)


City: New Delhi
Temperature: 36°C
Weather: Sunny
Humidity: 26%I now know the final answer.

Final Answer:
The capital of India is **New Delhi**. The current weather in New Delhi is **sunny**, with a temperature of **36°C** and a humidity of **26%**.

> Finished chain.


In [52]:
print(response["output"])

The capital of India is **New Delhi**. The current weather in New Delhi is **sunny**, with a temperature of **36°C** and a humidity of **26%**.
